# Analysis — game-agent

Read-only analysis of runs and learned latents. **Training stays in the `.py` scripts**; this notebook only *looks* at results.

1. Comparison curves from TensorBoard logs (`data/tb/`).
2. Latent diagnostics + probe (collapse / effective rank / nearest neighbours).

Run inside the container (Jupyter at http://localhost:8888).

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import torch
from autoencoder import Encoder

TB = '/app/data/tb'
MODELS = '/app/data/models'
FRAMES = '/app/data/frames'
%matplotlib inline

## 1. Comparison curves

SB3 names TB runs `PPO_1, PPO_2, ...` in creation order, so first list them with their final reward to identify which is which, then map labels below.

In [ ]:
from tensorboard.backend.event_processing import event_accumulator

def load_scalar(run_dir, tag='rollout/ep_rew_mean'):
    ea = event_accumulator.EventAccumulator(run_dir, size_guidance={'scalars': 0})
    ea.Reload()
    if tag not in ea.Tags().get('scalars', []):
        return [], []
    ev = ea.Scalars(tag)
    return [e.step for e in ev], [e.value for e in ev]

for r in sorted(glob.glob(os.path.join(TB, 'PPO_*')),
               key=lambda p: int(p.split('_')[-1])):
    s, v = load_scalar(r)
    if v:
        print(f'{os.path.basename(r):8s}  final={v[-1]:8.2f}  max_step={s[-1]}')

In [ ]:
# edit this map after reading the list above
RUNS = {
    # 'pixel':  'PPO_1',
    # 'latent': 'PPO_10',
    # 'SPR':    'PPO_13',
}

plt.figure(figsize=(8, 5))
for label, name in RUNS.items():
    s, v = load_scalar(os.path.join(TB, name))
    plt.plot(s, v, label=label)
plt.xlabel('steps'); plt.ylabel('ep_rew_mean')
plt.legend(); plt.grid(alpha=0.3); plt.title('Learning curves'); plt.show()

## 2. Latent diagnostics + probe

Label-free (no ball detection — respects *no human indication*). We check whether a latent is **informative** or **collapsed**:

- per-dim std (near 0 = collapsed),
- **effective dimensionality** (participation ratio over PCA spectrum),
- **nearest-neighbour grid**: a query frame and its closest frames in latent space — if the latent captures the ball/paddle, neighbours share their position.

In [ ]:
def load_encoder(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    enc = Encoder(ck['latent_dim']); enc.load_state_dict(ck['encoder']); enc.eval()
    return enc, ck['latent_dim']

def encode(enc, frames):
    x = torch.from_numpy(frames).float().div_(255.0).unsqueeze(1)
    with torch.no_grad():
        return enc(x).numpy()

frames = np.load(os.path.join(FRAMES, 'breakout.npz'))['frames']
enc, D = load_encoder(os.path.join(MODELS, 'breakout_encoder.pt'))  # Stage A (reconstruction)
Z = encode(enc, frames[:4000])
print('latent dim:', D, ' per-dim std (mean):', round(float(Z.std(0).mean()), 4))

In [ ]:
# PCA spectrum via SVD (no sklearn) + participation ratio = effective dimensionality
Zc = Z - Z.mean(0)
sv = np.linalg.svd(Zc, compute_uv=False)
ev = sv**2; ev = ev / ev.sum()
pr = (ev.sum()**2) / (ev**2).sum()
print(f'effective dimensionality (participation ratio): {pr:.1f} / {D}')
plt.plot(np.cumsum(ev), marker='.')
plt.xlabel('component'); plt.ylabel('cumulative explained variance')
plt.title('Latent spectrum'); plt.grid(alpha=0.3); plt.show()

In [ ]:
def nn_grid(enc, frames, queries=(123, 777, 2000, 3500), k=5, pool=5000):
    Z = encode(enc, frames[:pool])
    Zn = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)
    fig, axes = plt.subplots(len(queries), k + 1, figsize=(2 * (k + 1), 2 * len(queries)))
    for r, q in enumerate(queries):
        nn = np.argsort(-(Zn @ Zn[q]))[:k + 1]
        for col, idx in enumerate(nn):
            axes[r, col].imshow(frames[idx], cmap='gray'); axes[r, col].axis('off')
            axes[r, col].set_title('query' if col == 0 else 'nn', fontsize=8)
    plt.tight_layout(); plt.show()

nn_grid(enc, frames)  # swap in breakout_encoder_spr.pt to compare representations